# Data Validation: Required, Types, Rules, References, Schemas

## What you'll practice

Validation is where pipelines earn trust. You'll go from single checks to a declarative schema validator producing a data-quality report — the same ideas as Great Expectations,
Deequ (on Spark) and AWS Glue Data Quality, built from scratch.

> Function and class names in **Implement** are what the tests look for. Run all tests for this notebook with `pytest tests/etl/test_08_02_data_validation.py -v` from the lab root. Untouched exercises show as *skipped*.
>
> Solutions: `solutions/08_data_engineering_python/02_data_validation.ipynb`. Don't open it until you've tried.

### Setup — run this first (sample data and imports, not a solution)

In [ ]:
from pathlib import Path
LAB = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets").is_dir() and (p / "labkit").is_dir())   # lab root (works from any folder)
DATA = LAB / "datasets"
ORDERS_CSV = DATA / "csv" / "orders.csv"          # 5k+ messy orders
CUSTOMERS_CSV = DATA / "csv" / "customers.csv"
PRODUCTS_CSV = DATA / "csv" / "products.csv"
OUTPUT = LAB / "output" / "08"                       # scratch output folder (git-ignored)
OUTPUT.mkdir(parents=True, exist_ok=True)

## Exercise 1 — 🟢 Easy · Required fields

**Implement:** `missing_fields(record: dict, required: list[str]) -> list[str]`

Return required fields that are absent, `None`, or blank/whitespace strings, in the order given. `0`, `False` and `[]` are **present**.

**Input:** `record: dict, required: list[str]`

**Expected output:** `list[str]`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_missing_fields` in `tests/etl/test_08_02_data_validation.py`.

**Check:** `pytest tests/etl/test_08_02_data_validation.py -k test_missing_fields`

**YOUR CODE ↓**

## Exercise 2 — 🟡 Medium · Type checks against string data

**Implement:** `type_errors(record: dict, schema: dict[str, str]) -> dict[str, str]`

Raw CSV values are strings. `schema` maps field → `"int" | "float" | "bool" | "date" | "datetime" | "str"`. Return `{field: "expected <type>, got '<value>'"}` for values that
don't parse. Blank/missing values are **not** type errors (that's the required check's job). `bool` accepts true/false/1/0/yes/no (case-insensitive); `date` is `YYYY-MM-DD`;
`datetime` is ISO-8601 (with optional `Z`). Unknown type names → `ValueError`.

**Input:** `record: dict, schema: dict[str, str]`

**Expected output:** `dict[str, str]`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_type_errors_ok, test_type_errors_bad` in `tests/etl/test_08_02_data_validation.py`.

**Check:** `pytest tests/etl/test_08_02_data_validation.py -k "test_type_errors_ok or test_type_errors_bad"`

**YOUR CODE ↓**

## Exercise 3 — 🟡 Medium · A tiny rule engine

**Implement:** `evaluate_rules(records: list[dict], rules: dict[str, callable], id_field: str) -> dict`

`rules` maps a rule name to a predicate that returns `True` when the record **passes**. Return `{"total": n, "rules": {name: {"failed": count, "failed_ids": [first 5 ids], "pass_rate": float 4dp}}}`.
A predicate that raises counts as a failure for that record (bad data shouldn't crash validation).

**Input:** `records: list[dict], rules: dict[str, callable], id_field: str`

**Expected output:** `dict`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_evaluate_rules` in `tests/etl/test_08_02_data_validation.py`.

**Check:** `pytest tests/etl/test_08_02_data_validation.py -k test_evaluate_rules`

**YOUR CODE ↓**

## Exercise 4 — 🟠 Hard · Referential and cross-field validation

**Implement:** `validate_orders(orders: list[dict], customer_ids: set[str], catalog: dict[str, float], max_price_deviation: float = 0.5) -> list[tuple[str, list[str]]]`

For typed orders (`order_id`, `customer_id`, `product_id`, `unit_price`, `order_ts`, `updated_at`, `status`) return `(order_id, errors)` for every order with at least one error:
- `"unknown customer <id>"` / `"unknown product <id>"`
- `"price deviates <pct>% from catalog"` when `abs(unit_price - catalog) / catalog > max_price_deviation` (pct as int, rounded)
- `"updated_at before order_ts"`
- `"cancelled order with zero price"` is **not** an error; but `"delivered order with zero price"` is.
Keep the error order as listed. Then run it on the real orders (after your transform) with customers/products from `datasets/csv`.

**Input:** `orders: list[dict], customer_ids: set[str], catalog: dict[str, float], max_price_deviation: float = 0.5`

**Expected output:** `list[tuple[str, list[str]]]`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_validate_orders` in `tests/etl/test_08_02_data_validation.py`.

**Check:** `pytest tests/etl/test_08_02_data_validation.py -k test_validate_orders`

**YOUR CODE ↓**

## Challenge — 🔴 Very Hard · Declarative schema validator with a data-quality report · 🧩 Mini-project

**Implement:** `class SchemaValidator`

`SchemaValidator(spec)` where `spec = {"columns": {name: rules}}` and rules may include: `type` (`str|int|float|bool|date|datetime`), `required` (bool), `allowed` (list),
`min`/`max` (numbers or ISO strings for dates), `pattern` (regex, full match), `unique` (bool), `normalize` (`"upper"|"lower"|"strip"`, applied to strings before checks).
`validate(rows)` (rows of raw strings) returns `(valid_rows, rejects, report)`:
- `valid_rows`: typed values (int/float/bool/`date`/aware `datetime`); blank optional values → `None`; unknown extra columns dropped.
- `rejects`: `[{"row": 1-based index, "errors": ["<col>: <message>", ...], "raw": row}]`. Messages: `required`, `invalid <type>`, `not in allowed values`, `below min`, `above max`,
  `does not match pattern`, `duplicate value`.
- `report`: `{"rows": n, "valid": n, "rejected": n, "columns": {col: {"nulls": n, "errors": n}}}`.
Uniqueness: the **first** occurrence is valid, later ones are rejected. Build a spec for `datasets/csv/orders.csv` and look at the report.

**Input:** see the class description above

**Expected output:** an instance behaving as described

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_schema_validator` in `tests/etl/test_08_02_data_validation.py`.

**Check:** `pytest tests/etl/test_08_02_data_validation.py -k test_schema_validator`

**YOUR CODE ↓**